# 📓 GeoHierarchy End-to-End Example (Notebook Version)

In [ ]:
from pathlib import Path
import geopandas as gpd

from geohierarchy.core import GeoHierarchy
from geohierarchy.aggregation import Max, Sum, Mean
import os 

## 📁 Load data

In [ ]:
BASE_DIR = Path.cwd() / "test_files"

county = gpd.read_file(BASE_DIR / "county.gpkg")
tract = gpd.read_file(BASE_DIR / "tract.gpkg")
if os.path.isfile(BASE_DIR / "blockgroup.gpkg"):
    blockgroup = gpd.read_file(BASE_DIR / "blockgroup.gpkg")
else:
    blockgroup = None 

if os.path.isfile(BASE_DIR / "block.gpkg"):
    block = gpd.read_file(BASE_DIR / "block.gpkg")
else:
    block = None 

if os.path.isfile(BASE_DIR / "accessibility_streets.gpkg"):
    streets = gpd.read_file(BASE_DIR / "accessibility_streets.gpkg")
else:
    streets = None 

places = gpd.read_file(BASE_DIR / "accessibility_place.gpkg")

In [ ]:
gh = GeoHierarchy()
gh.add_level(
    "county",
    county[["population_total","geometry"]],
    Sum(),
    child="tract"
)
gh.add_level(
    "tract",
    tract[["vehicles_total","geometry"]],
    Sum(),
    parent="county",
)
if blockgroup is not None:
    gh.add_level(
        "blockgroup",
        blockgroup[["population_total","geometry"]],
        parent="tract",
        child="block"
    )
if block is not None:
    gh.add_level(
        "block",
        block[["population_total","geometry"]],
        parent="blockgroup"
    )
if streets is not None:
    gh.from_vector(
        streets[["accessibility","geometry"]],
        to_level="block",
        agg=Mean(weight_column="population_total"),
        injection_agg=Max(),
        buffer=10,
    )
gh.propagate() # Fills all columns
gh.add_level(
    "place",
    places[["accessibility","geometry"]],
    child="blockgroup"
)
gh.propagate() # Fills place columns
# Now any level has all columns "population_total", "vehicles_total" and "accessibility"

In [ ]:
m=gh["tract"][["accessibility","vehicles_total","population_total","geometry"]].explore(
    column="accessibility"
)
m